# Misión 4: El Cartógrafo de Patrones
### Semana 7 -- K-Means Clustering

**Dónde quedamos:** en la Misión 3 construiste un oráculo que, a partir de
lo que ya sabes, predice un número que no sabes -- **cuánto** va a valer
algo. Pero hay preguntas que no son "¿cuánto?". Son "¿qué tipo?" -- sin que
nadie te haya dicho de antemano cuáles son los tipos.

Hoy construyes esa otra herramienta: un mapa que agrupa datos parecidos
entre sí, sin que tú (ni el algoritmo) sepan de antemano qué grupos
existen. En este curso vas a ser su **cartógrafo**.


### Leyenda de iconos

| Icono | Acción | Qué significa |
|---|---|---|
| 👀 | **OBSERVA** | Ejecuta y observa -- no cambies nada |
| ✏️ | **MODIFICA** | Edita y vuelve a ejecutar |
| 🔮 | **PREDICE** | Escribe tu predicción en el cuadro de texto antes de ver el resultado -- participación, sin respuesta incorrecta (+1 XP) |
| 📝 | **ANOTA** | Escribe tu idea en el cuadro de texto -- participación, sin respuesta incorrecta (+1 XP) |
| 🧩 | **COMPLETA** | Reemplaza `___` por el valor correcto |
| 🔨 | **CONSTRUYE** | Escribe código desde cero en el bloque "Tu código aquí" |
| 🔧 | **DEBUG** | Ejecuta, lee el error, corrígelo |
| ✅ | **VERIFICA** | El autograder revisa tu respuesta |
| ❓ | **TEORÍA** | Pregunta de opción múltiple (se muestra al ejecutar la celda) |
| 💭 | **REFLEXIONA** | Respuesta abierta -- calificada por IA, feedback instantáneo (+5 XP) |

---


In [ ]:
# Carga el autograder y el dataset de esta lección
!wget -q "https://raw.githubusercontent.com/Santa-Maria-de-los-Andes/intro_to_stats/main/course-python-stats/Week%207/autograder_nb6.py"
!wget -q "https://raw.githubusercontent.com/Santa-Maria-de-los-Andes/intro_to_stats/main/course-python-stats/Week%207/spotify_es.csv"
from autograder_nb6 import Autograder
grader = Autograder()

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df_canciones = pd.read_csv('spotify_es.csv')
df_canciones = df_canciones.dropna(subset=['Artistas', 'Nombre_Cancion']).copy()
df_canciones = df_canciones[df_canciones['Tempo'] > 0].copy()  # unas pocas filas con Tempo=0 son fallos de deteccion, no canciones reales de 0 BPM

print(f"Canciones cargadas: {len(df_canciones)}")

## 👀 Antes de cualquier código: ¿cuántos grupos ves?

Mira el siguiente gráfico. Cada punto es una canción real de Spotify: en
el eje X, qué tan **bailable** es (de 0 a 1); en el eje Y, su **tempo**
(BPM -- beats por minuto, qué tan rápida es). No hay colores ni etiquetas
todavía -- son solo puntos.

**No te preocupes por el código todavía** -- solo mira el gráfico y
piensa: si tuvieras que separar estas canciones en grupos "parecidas entre
sí", ¿cuántos grupos usarías? ¿en qué te fijarías para decidir?


In [ ]:
# 👀 OBSERVA -- no cambies nada, solo mira el resultado
plt.figure(figsize=(7, 5))
plt.scatter(df_canciones['Bailabilidad'], df_canciones['Tempo'], alpha=0.15, s=8)
plt.xlabel('Bailabilidad (0 a 1)')
plt.ylabel('Tempo (BPM)')
plt.title(f'{len(df_canciones)} canciones -- sin agrupar todavía')
plt.show()

🔮 Antes de seguir leyendo: ¿cuántos grupos "parecidos entre sí" crees que
hay en este gráfico? Ejecuta la celda de abajo y escribe tu predicción --
no hay respuesta incorrecta, ni siquiera un algoritmo te va a dar una
única respuesta "correcta".


In [ ]:
# 🔮 PREDICE -- ejecuta esta celda para escribir tu predicción
grader.check_cuantos_grupos()

---
## 🧠 Concepto 1 -- ¿Qué hace k-means?

k-means es un algoritmo que agrupa puntos parecidos **sin que nadie le
diga de antemano** qué significa cada grupo. Así es como decide:

1. Coloca **k** "centros" (centroides) en el espacio de tus datos -- al
   azar, para empezar.
2. Cada punto se asigna al centro **más cercano**.
3. Cada centro se recalcula como el **promedio** de los puntos que le
   tocaron.
4. Repite los pasos 2 y 3 hasta que las asignaciones casi no cambian.

Fíjate en lo que **nunca** usa: ninguna etiqueta real (como el género de
una canción). Solo usa la posición de los puntos. Tú le dices cuántos
grupos quieres (k) -- el algoritmo decide cuáles son.


In [ ]:
# 📝 ANOTA -- ejecuta esta celda para escribir tu idea
grader.check_como_agrupa()

---
## ✨ Ejemplo guiado -- tu primer mapa (sin escalar)

Vas a pedirle a k-means que agrupe las canciones en k=3 grupos, usando
`Bailabilidad` y `Tempo` -- las mismas dos columnas del scatter de arriba.
Todavía no te preocupes por nada más: solo mira qué hace.


In [ ]:
# 👀 OBSERVA -- k-means sobre los datos SIN escalar
X_demo = df_canciones[['Bailabilidad', 'Tempo']]

modelo_demo_raw = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_demo)
df_canciones['cluster_demo_raw'] = modelo_demo_raw.labels_

plt.figure(figsize=(7, 5))
plt.scatter(df_canciones['Bailabilidad'], df_canciones['Tempo'],
            c=df_canciones['cluster_demo_raw'], cmap='viridis', alpha=0.2, s=8)
plt.xlabel('Bailabilidad (0 a 1)')
plt.ylabel('Tempo (BPM)')
plt.title('k=3, SIN escalar -- ¿qué separó realmente a estos grupos?')
plt.show()

print(df_canciones.groupby('cluster_demo_raw')[['Bailabilidad', 'Tempo']].mean())

Fíjate en la tabla de arriba: entre los tres grupos, `Tempo` cambia
muchísimo (de ≈91 a ≈167 BPM) mientras que `Bailabilidad` casi no se mueve
(de ≈0.50 a ≈0.61) -- los colores del gráfico probablemente forman
**bandas horizontales**, separadas casi solo por `Tempo`. Antes de
explicar por qué, mira los rangos reales de cada columna.


In [ ]:
# 👀 OBSERVA -- los rangos reales de cada columna
print("Rango real de Bailabilidad:", df_canciones['Bailabilidad'].min(), "a", df_canciones['Bailabilidad'].max())
print("Rango real de Tempo:       ", df_canciones['Tempo'].min(), "a", df_canciones['Tempo'].max())
print()
print("Promedio de Bailabilidad:", round(df_canciones['Bailabilidad'].mean(), 3))
print("Promedio de Tempo:       ", round(df_canciones['Tempo'].mean(), 1))

---
## 🧠 Concepto 2 -- Por qué escalar es un no-negociable

`Bailabilidad` vive en el rango 0 a 1. `Tempo` vive en el rango de las
decenas a los cientos (≈30-243 BPM, promedio ≈122). k-means mide qué tan
"cerca" está un punto de otro con **distancia** -- y una diferencia de 50
BPM ya vale 50, mientras que la diferencia **máxima posible** de
Bailabilidad es 1.

Eso significa que, sin hacer nada especial, `Tempo` domina casi por
completo la distancia -- y k-means termina agrupando casi solo por
`Tempo`, dejando que `Bailabilidad` apenas influya. Esto no es un detalle
menor: **es el error más común al usar k-means**, y la solución es
sencilla -- escalar cada columna antes de ajustar, con `StandardScaler`
(resta el promedio y divide entre la desviación estándar, dejando cada
columna con promedio 0 y desviación 1).


### Repite el ejemplo, esta vez escalando primero


In [ ]:
# 👀 OBSERVA -- las mismas dos columnas, esta vez escaladas antes de ajustar
X_escalado_demo = StandardScaler().fit_transform(X_demo)

modelo_demo_escalado = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_escalado_demo)
df_canciones['cluster_demo_escalado'] = modelo_demo_escalado.labels_

plt.figure(figsize=(7, 5))
plt.scatter(df_canciones['Bailabilidad'], df_canciones['Tempo'],
            c=df_canciones['cluster_demo_escalado'], cmap='viridis', alpha=0.2, s=8)
plt.xlabel('Bailabilidad (0 a 1)')
plt.ylabel('Tempo (BPM)')
plt.title('k=3, escalado -- ahora Bailabilidad sí influye')
plt.show()

print(df_canciones.groupby('cluster_demo_escalado')[['Bailabilidad', 'Tempo']].agg(['mean', 'count']))

---
## 🧠 Concepto 3 -- Las etiquetas de cluster son arbitrarias hasta que TÚ las interpretas

Fíjate en la tabla de arriba: ahora los tres clusters tienen tamaños
comparables, y se distinguen tanto por `Bailabilidad` **como** por
`Tempo` -- uno agrupa canciones muy bailables de tempo medio, otro
canciones poco bailables y rápidas, otro canciones poco bailables y
lentas. Pero el **número** que le tocó a cada cluster (0, 1 o 2) es
completamente arbitrario: si vuelves a ejecutar k-means en otra
computadora, esos mismos grupos de canciones pueden terminar con otros
números.

Lo único con significado real son los puntos que quedaron agrupados y sus
centroides -- **nunca el índice**. Interpretar un cluster significa
describir qué tienen en común sus puntos, en tus propias palabras y con
tus propios números -- no inventarle un nombre "oficial".

Una pregunta natural: si agrupas canciones por bailabilidad y tempo... ¿los
clusters terminan siendo, básicamente, los géneros de la canción (Pop,
Rock, Reggaetón...)? Este dataset tiene **114 géneros reales** distintos
-- vamos a comprobarlo con datos.


### Comprueba: ¿los clusters se parecen a los géneros reales?

`spotify_es.csv` sí tiene una columna `Genero` con el género real de cada
canción (114 géneros distintos: Pop, Rock, Reggaetón, K-Pop, Jazz...) --
pero nunca se la diste a k-means como predictor. Si el género no tuviera
ninguna relación con `Bailabilidad`/`Tempo`, cada género representaría
apenas ≈1/114 ≈ 0.9% de cualquier cluster al azar. Compara eso con lo que
realmente aparece.


In [ ]:
# 👀 OBSERVA -- el género MÁS COMÚN dentro de cada cluster, como % de ese cluster
for c in sorted(df_canciones['cluster_demo_escalado'].unique()):
    sub = df_canciones[df_canciones['cluster_demo_escalado'] == c]
    top_genero = sub['Genero'].value_counts(normalize=True).head(1)
    print(f"Cluster {c} (n={len(sub)}): género más común = {top_genero.index[0]} "
          f"({top_genero.iloc[0] * 100:.1f}% de este cluster)")

#### 🧩 Completa -- ¿los clusters SON géneros reales? (5 pts)

Mira los porcentajes que imprimiste arriba (todos rondan apenas el 2-3%,
con 114 géneros posibles). Completa:
- `respuesta_cluster_es_genero`: escribe `"si"` o `"no"`
- `explicacion_cluster_es_genero`: en 1 frase, cita uno de los números de
  arriba para justificar tu respuesta


In [ ]:
# 🧩 COMPLETA

respuesta_cluster_es_genero = "___"
explicacion_cluster_es_genero = "___"

print(f"Respuesta: {respuesta_cluster_es_genero}")

In [ ]:
grader.check_cluster_es_genero()

---
## ✨ Muestra -- ¿qué cambia si agregas una tercera variable?

Antes de trazar tu propio mapa, mira qué pasa si al modelo de arriba
(`Bailabilidad` + `Tempo`, escalado, k=3) le agregas `Energia` -- la
tercera columna del menú que vas a usar en el Ejercicio 1.


In [ ]:
# 👀 OBSERVA -- mismo k=3, esta vez con Bailabilidad + Tempo + Energia
X_demo3 = df_canciones[['Bailabilidad', 'Tempo', 'Energia']]
X_demo3_escalado = StandardScaler().fit_transform(X_demo3)

modelo_demo3 = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_demo3_escalado)
df_canciones['cluster_demo3'] = modelo_demo3.labels_

print(df_canciones.groupby('cluster_demo3')[['Bailabilidad', 'Tempo', 'Energia']].agg(['mean', 'count']))

#### 🔍 Un patrón que quizás no esperabas

Agregar `Energia` sí cambia el agrupamiento de forma notoria -- ni total,
ni trivial (Adjusted Rand Index ≈0.53 frente al mapa de solo 2 variables).
`Energia` está débilmente correlacionada con `Bailabilidad` y `Tempo`
(r≈0.13 y r≈0.24), así que aporta información genuinamente nueva: el mapa
se redibuja en parte en vez de quedar casi igual. Tenlo en mente al elegir
tus predictores para el Ejercicio 1.

---
#### ✅ Ejercicio 1 -- Traza tu mapa (40 pts)

🔨 Elige **2 o 3** predictores de este menú: `Bailabilidad`, `Tempo`,
`Energia` (puedes incluir `Energia`, como en la muestra de arriba, o
quedarte con las 2 del ejemplo guiado).

Pasos:
1. Guarda tu elección en `predictores_ex1` (lista de 2 o 3 nombres)
2. Escala esas columnas con `StandardScaler`
3. Ajusta `KMeans(n_clusters=3, random_state=42)` sobre los datos
   **escalados** -- guarda el modelo en `modelo_ex1`
4. Elige un cluster para interpretar y guarda su número (0, 1 o 2) en
   `cluster_interes_ex1`
5. Escribe 1-2 oraciones interpretando el centroide de ese cluster en
   `interpretacion_ex1` -- describe el grupo con tus propias variables/
   números, sin afirmar que ES un género real (como practicaste arriba)

Variables que espera el autograder: `predictores_ex1` (lista de 2 o 3),
`modelo_ex1` (KMeans ya ajustado), `cluster_interes_ex1` (entero),
`interpretacion_ex1` (string).


In [ ]:
# 🔨 CONSTRUYE

# ============================
#      Tu código aquí
# ============================


print(f"Predictores: {predictores_ex1}")
print(f"Centroide del cluster {cluster_interes_ex1}: {modelo_ex1.cluster_centers_[cluster_interes_ex1]}")

In [ ]:
grader.check_ex1()

In [ ]:
# ✅ CHECKPOINT -- necesitas 80% en esta sección para continuar
grader.check_mini_a()

#### 💭 Reflexiona (respuesta abierta -- calificada por IA, +5 XP)

Arriba viste qué pasó al agregar `Energia` a `Bailabilidad`+`Tempo` (un
cambio moderado, ni total ni trivial). Compara eso con tu propio
Ejercicio 1.


In [ ]:
# 💭 Reflexiona -- ejecuta esta celda para responder
grader.check_reflexion_ex2()

---
## 🧠 Concepto 4 -- ¿Por qué importa que un agrupamiento generalice?

k-means no tiene una "respuesta correcta" que pueda fallar -- no hay un
Puntaje real (como en la Misión 3) contra el cual medir un error. Pero eso
no significa que la calidad de tu mapa no se pueda poner a prueba.

Imagina dos usos reales de estos clusters:

- **Recomendación**: cuando termina una canción, Spotify recomienda otra
  del mismo cluster ("canciones parecidas a esta"). Si mañana sube una
  canción nueva -- que tu modelo *nunca vio* -- y la asignas a un cluster
  con `.predict()`, ¿sus compañeras de cluster de verdad se le parecen?
- **Segmentación**: una discográfica quiere usar tus clusters para decidir
  presupuesto de marketing por "tipo de canción". Si vuelves a ajustar el
  modelo con otra muestra de canciones y los grupos se ven completamente
  distintos, esa segmentación no sirve para tomar una decisión real.

En ambos casos la pregunta es la misma que la Misión 3 te enseñó a hacer
con regresión: **¿el patrón que encontraste se sostiene en datos que el
modelo nunca vio, o fue solo un accidente de la muestra que usaste para
ajustar?** La forma de comprobarlo es la misma herramienta de siempre --
train/test -- aplicada a un algoritmo que nunca tuvo una "respuesta
correcta" para empezar.


### Repite el ejemplo, esta vez separando entrenamiento y prueba

Vas a repetir el ejemplo guiado de siempre (`Bailabilidad` + `Tempo`,
escalado, k=3) -- pero esta vez ajustas k-means **solo** con el 90% de
entrenamiento, y usas `.predict()` (nunca `.fit()` de nuevo) para asignarle
un cluster a las canciones de prueba, que el modelo nunca vio al ajustar.


In [ ]:
# 👀 OBSERVA -- separa train/test, escala SOLO con train, ajusta SOLO con train
from sklearn.model_selection import train_test_split

train_demo, test_demo = train_test_split(df_canciones, test_size=0.1, random_state=42)

escalador_demo = StandardScaler().fit(train_demo[['Bailabilidad', 'Tempo']])
X_train_demo = escalador_demo.transform(train_demo[['Bailabilidad', 'Tempo']])
X_test_demo  = escalador_demo.transform(test_demo[['Bailabilidad', 'Tempo']])

modelo_demo_generaliza = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_train_demo)

train_demo = train_demo.copy()
test_demo  = test_demo.copy()
train_demo['cluster'] = modelo_demo_generaliza.labels_
test_demo['cluster']  = modelo_demo_generaliza.predict(X_test_demo)

print("Perfil por cluster -- ENTRENAMIENTO (90%, lo que el modelo vio al ajustar):")
print(train_demo.groupby('cluster')[['Bailabilidad', 'Tempo']].agg(['mean', 'count']))
print()
print("Perfil por cluster -- PRUEBA (10%, canciones que el modelo NUNCA vio):")
print(test_demo.groupby('cluster')[['Bailabilidad', 'Tempo']].agg(['mean', 'count']))

Fíjate: los promedios de `Bailabilidad`/`Tempo` por cluster son casi
idénticos entre entrenamiento y prueba (y las proporciones de tamaño
también se parecen). Eso es lo que significa que un agrupamiento
**generalice**: la estructura que encontraste no es un accidente de qué
canciones cayeron en tu muestra de ajuste -- se sostiene en canciones que
el modelo nunca vio. Si los perfiles fueran muy distintos entre train y
test, cualquier recomendación o segmentación basada en ese mapa sería poco
confiable.


#### 💭 Reflexiona (respuesta abierta -- calificada por IA, +5 XP)

Piensa en los dos usos reales del Concepto 4: **recomendación** (sugerir
la próxima canción del mismo cluster) y **segmentación** (decidir
presupuesto de marketing por "tipo de canción"). En el ejemplo guiado de
arriba viste que los perfiles de cluster casi no cambian entre
entrenamiento y prueba.


In [ ]:
# 💭 Reflexiona -- ejecuta esta celda para responder
grader.check_reflexion_generalizacion()

---
## ❓ Quiz de cierre -- 5 preguntas de teoría (25 pts)

Antes de la reflexión final, confirma que las ideas clave quedaron claras.


In [ ]:
# ❓ Pregunta t1 -- ejecuta esta celda para verla y responder
grader.check_t1()

In [ ]:
# ❓ Pregunta t2 -- ejecuta esta celda para verla y responder
grader.check_t2()

In [ ]:
# ❓ Pregunta t3 -- ejecuta esta celda para verla y responder
grader.check_t3()

In [ ]:
# ❓ Pregunta t4 -- ejecuta esta celda para verla y responder
grader.check_t4()

In [ ]:
# ❓ Pregunta t5 -- ejecuta esta celda para verla y responder
grader.check_t5()

---
## 🪞 Reflexión final (respuesta abierta -- calificada por IA, +5 XP)

Ya usaste k-means para agrupar canciones por variables numéricas, sin
decirle al algoritmo nada sobre el género real de cada canción.


In [ ]:
# 💭 Reflexiona -- ejecuta esta celda para responder
grader.check_reflexion_final()

In [ ]:
# ✅ CHECKPOINT -- necesitas 80% en esta sección para continuar
grader.check_mini_b()

---
#### 🎯 Reto (bonus, +10 XP) -- Cambia el número de regiones

Elige un valor de k **distinto de 3** (por ejemplo, 2, 4 o 5) y vuelve a
ajustar k-means -- reusa los mismos predictores escalados de tu Ejercicio
1. Guarda tu elección en `k_reto1`, el modelo en `modelo_reto1`, y en
una frase describe qué cambió en `comentario_reto1` (¿algún grupo se
dividió? ¿se fusionaron dos?).


In [ ]:
# 🔨 CONSTRUYE (bonus)

# ============================
#      Tu código aquí
# ============================


print(f"k_reto1 = {k_reto1}")
print(f"Comentario: {comentario_reto1}")

In [ ]:
grader.check_reto1()

---
## 🏁 Fin de la Misión 4 -- Semana 7

Construiste dos mapas de clusters sobre el mismo dataset de canciones de
Spotify: uno sin escalar (dominado por una sola variable) y otro escalado
(donde todas las variables pesan). Aprendiste que k-means no usa ninguna
etiqueta real -- agrupa por posición, no por significado -- y que
interpretar un cluster es tu trabajo, no algo que el algoritmo hace por
ti.

La próxima vez que alguien te diga "el algoritmo encontró 3 tipos de
clientes/canciones/lo-que-sea", ya sabes qué preguntar: ¿escalaron las
variables antes de agrupar? ¿qué tan estable es ese número de grupos?
¿esos "tipos" describen algo real, o son solo un número que el algoritmo
les puso?


In [ ]:
grader.resumen()